# Main study: Base, RAG, CAG and QLoRA on TAT-QA

One notebook for the whole experiment. Run the cells **from top to bottom**.

**Before you start**
- **Google Colab:** Runtime → Change runtime type → **T4 GPU** → Save.
- **Kaggle** (free 30 GPU hours a week): Settings panel on the right → Accelerator → **GPU T4 x2**, and Internet → **On**. (Your Kaggle account needs a verified phone number for both.) Do not choose P100.

**Good to know**
- Every model and strategy saves to its **own file** (`results/raw/<model>__<strategy>.csv`). Nothing is overwritten.
- Every run cell **continues where it stopped**. Running a finished cell again just reports *0 to run*.
- **Progress is saved to GitHub automatically** about every 5 minutes while a model runs, and when it finishes. If the session disconnects (the machine is wiped), at most the last few minutes are lost.
- **After a disconnect:** run the GPU check, *Setup A, B, C* and *Step 1* again, then the cell you were on. It downloads the saved results and continues from the next unanswered question.
- Do **not** use File → Save a copy in GitHub. Use *Step 7* to save results.

In [ ]:
# Works on Colab and Kaggle. You should see 'Tesla T4'.
# If it says NO GPU: Colab -> Runtime -> Change runtime type -> T4 GPU;
#                    Kaggle -> Settings -> Accelerator -> GPU T4 x2.
import os
ON_KAGGLE = os.path.exists('/kaggle')
PROJECT = '/kaggle/working/project' if ON_KAGGLE else '/content/project'
# Kaggle gives two T4s; use only one so speed and memory match a single T4 everywhere.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
!nvidia-smi -L
import torch
print('Platform:', 'Kaggle' if ON_KAGGLE else 'Colab')
print('GPU ready:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - change the accelerator first')

## Setup A. Get the code and the results saved so far
Your GitHub token (starts with `ghp_`) is kept **only for this session**, so results can be saved to GitHub automatically. It is never written to a file.

**Kaggle tip:** instead of pasting it every time, add it once under *Add-ons → Secrets* with the name `GITHUB_TOKEN` (and your Hugging Face token as `HF_TOKEN`), and tick both for this notebook.

In [ ]:
import getpass, os
REPO = "github.com/faheemjabbar39-hub/llm-knowledge-integration.git"
def secret(name):
    """Kaggle Secret if one exists, otherwise ask."""
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return getpass.getpass(f"{name}: ")
os.environ['GITHUB_TOKEN'] = gh_token = secret('GITHUB_TOKEN').strip()
if not os.path.exists(PROJECT):
    !git clone -q https://{gh_token}@{REPO} {PROJECT}
    !git -C {PROJECT} remote set-url origin https://{REPO}
else:
    !git -C {PROJECT} pull -q --rebase --autostash -X theirs https://{gh_token}@{REPO} main
%cd {PROJECT}
!git log --oneline -1
print('Results already saved:')
!ls results/raw

## Setup B. Install packages and test the code (about 4 minutes)
The last line should say **31 passed**.

In [ ]:
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pytest -q 2>&1 | tail -2

## Setup C. Log in to Hugging Face
Paste your Hugging Face **Read** token (it starts with `hf_`). Llama needs the access you already have; Mistral and Qwen are open to everyone.

In [ ]:
from huggingface_hub import login
login(token=secret('HF_TOKEN').strip())

## Step 1. Prepare the data (1 minute)
Downloads TAT-QA and builds the files. The official splits are kept:
- **train** → used only to train QLoRA
- **dev** → used only for the checks in Step 2
- **test** → 504 questions used for all final results

The test report extracts are put into 3 groups ("blocks") of 28. Each block is one small knowledge base: RAG searches it, CAG preloads it.

In [ ]:
!python scripts/01_prepare_tatqa.py

## Step 2. Check the setup (about 3 minutes)
**A.** How often does search find the right report extract? (needs to be above 60%, ideally above 80%)

**B.** Does every block fit in each model's memory limit?

Both must say **GOOD / OK** before you continue. If not, stop and ask.

In [ ]:
!python scripts/02_check_setup.py

## Step 3. Smoke test (about 10 minutes)
30 questions through Base, RAG and CAG on Llama-3.1-8B, to confirm everything runs. These 30 answers are kept, so the full run in Step 4 does not repeat them.

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy base rag cag --limit 30

In [ ]:
# Look at a few answers from each strategy.
import pandas as pd
pd.set_option('display.max_colwidth', 70)
import os
for s in ['base', 'rag', 'cag']:
    path = f'results/raw/llama8b__{s}.csv'
    if not os.path.exists(path):
        print(f'{s.upper()}: no results yet - the cell above did not finish. Read its last lines for the reason.')
        continue
    d = pd.read_csv(path)
    print(f"\n{s.upper()}: {d.correct.mean():.0%} correct, {d.hallucinated.mean():.0%} wrong answers, "
          f"{d.abstained.mean():.0%} said I don't know, median {d.latency_ms.median()/1000:.1f}s")
    display(d[['question', 'gold', 'final_answer', 'correct']].head(6))

## Step 4. Base, RAG and CAG: one cell per model
Each cell loads the model once and runs the three strategies on all 504 test questions (roughly 1.5 to 2 hours per model on a T4). Run **Step 7** after each model to save the results.

### 4a. Llama-3.1-8B

In [ ]:
!python scripts/03_run_eval.py --model llama8b --strategy base rag cag

### 4b. Mistral-7B

In [ ]:
!python scripts/03_run_eval.py --model mistral7b --strategy base rag cag

### 4c. Qwen2.5-7B

In [ ]:
!python scripts/03_run_eval.py --model qwen7b --strategy base rag cag

### 4d. Results of Base, RAG and CAG
Accuracy, hallucination, speed, memory, cost, results by kind of question, and significance tests. Works with whatever has finished so far.

In [ ]:
# ===================== Step 4 results: how good are Base, RAG and CAG? =====================
# Paste this after cells 4a, 4b, 4c. It reads whatever has finished in results/raw/,
# so you can run it at any time (even with only one model done).
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from kis.stats import binary_tests

TOTAL = 504                                    # test questions per model and strategy
ORDER_S = ["base", "rag", "cag", "qlora"]
ORDER_M = ["llama8b", "mistral7b", "qwen7b"]
NAMES_S = {"base": "Base", "rag": "RAG", "cag": "CAG", "qlora": "QLoRA"}
NAMES_M = {"llama8b": "Llama-3.1-8B", "mistral7b": "Mistral-7B", "qwen7b": "Qwen2.5-7B"}
GPU_USD_PER_HOUR = 0.35                        # same price as in configs/experiment.yaml
IDK = "Said I don't know %"

files = sorted(glob.glob("results/raw/*.csv"))
if not files:
    raise SystemExit("No results yet: run 4a, 4b or 4c first.")
df = pd.concat([pd.read_csv(f, dtype={"tag": "string"}) for f in files], ignore_index=True)
df["tag"] = df["tag"].fillna("")
df = df[df["tag"] == ""]                       # main conditions only (extra runs such as --no-docs are left out)
peak = {}
for f in files:
    m = f.replace(".csv", ".meta.json")
    if os.path.exists(m):
        meta = json.load(open(m))
        peak[(meta["model"], meta["strategy"])] = meta.get("peak_gpu_gb")

def order(df, col, order_list):
    return df.assign(_o=df[col].map({k: i for i, k in enumerate(order_list)}).fillna(99)).sort_values("_o").drop(columns="_o")

# ---------- 1. Progress ----------
prog = df.groupby(["model", "strategy"]).size().unstack().reindex(
    index=[m for m in ORDER_M if m in df.model.unique()], columns=[s for s in ORDER_S if s in df.strategy.unique()])
display(Markdown(f"### 1. Progress: questions answered (out of {TOTAL})"))
display(prog.fillna(0).astype(int).rename(index=NAMES_M, columns=NAMES_S))
if (prog.fillna(0) < TOTAL).any().any():
    print("Some runs are not finished yet: numbers below are provisional until every cell shows 504.")

# ---------- 2. Main results table ----------
g = df.groupby(["model", "strategy"])
t = pd.DataFrame({
    "Questions": g.size(),
    "Accuracy (strict) %": g["correct"].mean() * 100,
    "Accuracy (lenient) %": g["contains"].mean() * 100,
    "F1 (text answers)": g["f1"].mean(),
    "Hallucination %": g["hallucinated"].mean() * 100,
    IDK: g["abstained"].mean() * 100,
    "Search found right extract %": g["retrieval_hit"].mean() * 100,
    "Median time (s)": g["latency_ms"].median() / 1000,
    "Slow answers p95 (s)": g["latency_ms"].quantile(.95) / 1000,
    "Prompt length (tokens)": g["prompt_tokens"].mean(),
}).reset_index()
t["Peak GPU memory (GB)"] = [peak.get((m, s)) for m, s in zip(t.model, t.strategy)]
t["Cost per 1,000 questions ($)"] = GPU_USD_PER_HOUR * (g["latency_ms"].mean().values / 1000) * 1000 / 3600
t = order(order(t, "strategy", ORDER_S), "model", ORDER_M)
t["Model"], t["Strategy"] = t.model.map(NAMES_M), t.strategy.map(NAMES_S)
main = t.drop(columns=["model", "strategy"]).set_index(["Model", "Strategy"])
display(Markdown("### 2. Main results (all test questions)"))
display(main.style.format({c: "{:.1f}" for c in main.columns if "%" in c} | {
    "F1 (text answers)": "{:.3f}", "Median time (s)": "{:.2f}", "Slow answers p95 (s)": "{:.2f}",
    "Prompt length (tokens)": "{:,.0f}", "Peak GPU memory (GB)": "{:.1f}", "Cost per 1,000 questions ($)": "{:.3f}",
    "Questions": "{:,.0f}"}, na_rep="-"))

# ---------- 3. Accuracy by kind of question ----------
type_names = {"span": "Value/words from report", "multi-span": "Several values",
              "arithmetic": "Calculation", "count": "Counting"}
source_names = {"table": "Answer in table", "text": "Answer in text", "table-text": "Needs table + text"}
for col, names, title in [("answer_type", type_names, "kind of answer"), ("answer_from", source_names, "where the answer is")]:
    p = (df.groupby(["model", "strategy", col])["correct"].mean() * 100).unstack(col)
    p = p.reindex(columns=[c for c in names if c in p.columns]).rename(columns=names)
    p = p.reset_index()
    p = order(order(p, "strategy", ORDER_S), "model", ORDER_M)
    p["Model"], p["Strategy"] = p.model.map(NAMES_M), p.strategy.map(NAMES_S)
    p = p.drop(columns=["model", "strategy"]).set_index(["Model", "Strategy"])
    counts = df.drop_duplicates("item_id")[col].map(names).value_counts()
    display(Markdown(f"### 3. Accuracy (strict, %) by {title}"))
    print("Questions per group:", ", ".join(f"{k} {v}" for k, v in counts.items()))
    display(p.style.format("{:.1f}", na_rep="-"))

# ---------- 4. Are the differences real? (paired McNemar tests, Holm-corrected) ----------
display(Markdown("### 4. Are the differences real or chance? (same questions compared one by one)"))
rows = []
for model in [m for m in ORDER_M if m in df.model.unique()]:
    sub = df[df.model == model]
    if sub.strategy.nunique() < 2:
        continue
    for metric, label in [("correct", "Accuracy"), ("hallucinated", "Hallucination")]:
        res = binary_tests(sub, metric)
        omni = res.get("omnibus")
        for r in res.get("pairwise", []):
            a, b, diff = r["a"], r["b"], r["diff_pp"]
            if ORDER_S.index(a) < ORDER_S.index(b):          # always "newer strategy vs older", e.g. RAG vs Base
                a, b, diff = b, a, -diff
            rows.append({"Model": NAMES_M[model], "Measure": label,
                         "Comparison": f"{NAMES_S[a]} vs {NAMES_S[b]}",
                         "Difference (points)": diff, "Paired questions": res["n_items"],
                         "p (Holm)": r["p_holm"], "Real difference?": "YES" if r["significant"] else "no",
                         "Overall test (Cochran's Q) p": omni["p"] if omni else None})
if rows:
    display(pd.DataFrame(rows).set_index(["Model", "Measure", "Comparison"]).style.format(
        {"Difference (points)": "{:+.1f}", "p (Holm)": "{:.4f}", "Overall test (Cochran's Q) p": "{:.4f}"}, na_rep="-"))
    print("Difference = first minus second, in percentage points (e.g. +40 = 40 points higher than Base).")
    print("'YES' means p < 0.05 after Holm correction: very unlikely to be chance.")
else:
    print("Needs at least two finished strategies for the same model.")

# ---------- 5. Charts ----------
COLORS = {"base": "#2a78d6", "rag": "#1baf7a", "cag": "#eda100", "qlora": "#eb6834"}
models = [m for m in ORDER_M if m in df.model.unique()]
strats = [s for s in ORDER_S if s in df.strategy.unique()]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, (col, title) in zip(axes, [("correct", "Accuracy (strict), higher is better"),
                                   ("hallucinated", "Hallucination, lower is better"),
                                   ("latency_ms", "Median time per answer (s), lower is better")]):
    x, w = np.arange(len(models)), 0.8 / len(strats)
    for i, s in enumerate(strats):
        if col == "latency_ms":
            vals = [df[(df.model == m) & (df.strategy == s)][col].median() / 1000 for m in models]
        else:
            vals = [df[(df.model == m) & (df.strategy == s)][col].mean() * 100 for m in models]
        bars = ax.bar(x + (i - (len(strats) - 1) / 2) * w, vals, w * 0.92, color=COLORS[s], label=NAMES_S[s])
        for b, v in zip(bars, vals):
            if not np.isnan(v):
                ax.annotate(f"{v:.1f}" if col == "latency_ms" else f"{v:.0f}", (b.get_x() + b.get_width() / 2, v),
                            xytext=(0, 2), textcoords="offset points", ha="center", fontsize=8)
    ax.set_xticks(x, [NAMES_M[m] for m in models])
    ax.set_title(title, loc="left", fontsize=11, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_ylabel("seconds" if col == "latency_ms" else "% of questions")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, frameon=False, ncol=len(strats), loc="upper center", bbox_to_anchor=(0.5, 1.0))
plt.tight_layout(rect=(0, 0, 1, 0.92))
os.makedirs("results/figures", exist_ok=True)
plt.savefig("results/figures/step4_overview.png", dpi=150)
plt.show()

# ---------- 6. Plain-language summary ----------
display(Markdown("### 6. Summary in plain words"))
for model in models:
    sub = t[t.model == model].set_index("strategy")
    if "base" not in sub.index:
        continue
    base = sub.loc["base"]
    line = [f"**{NAMES_M[model]}**: Base answers {base['Accuracy (strict) %']:.1f}% correctly "
            f"(hallucination {base['Hallucination %']:.1f}%, said I don't know {base[IDK]:.1f}%)."]
    for s in ["rag", "cag", "qlora"]:
        if s in sub.index:
            r = sub.loc[s]
            line.append(f"{NAMES_S[s]}: {r['Accuracy (strict) %']:.1f}% correct "
                        f"({r['Accuracy (strict) %'] - base['Accuracy (strict) %']:+.1f} points vs Base), "
                        f"hallucination {r['Hallucination %']:.1f}%, {r['Median time (s)']:.1f}s per answer.")
    best = sub["Accuracy (strict) %"].idxmax()
    line.append(f"Most accurate so far: **{NAMES_S[best]}**.")
    display(Markdown(" ".join(line)))

main.to_csv("results/step4_summary.csv")
print("\nSaved: results/step4_summary.csv and results/figures/step4_overview.png (Step 7 uploads them to GitHub).")


## Step 5. QLoRA: one cell per model
Each cell does three things:
1. **Trains** small adapter layers on 1,202 TAT-QA *training* questions (roughly 1 to 1.5 hours on a T4). If the adapter already exists, this is skipped.
2. **Tests** the fine-tuned model on the 504 test questions, with the same documents RAG finds.
3. **Extra check:** tests it again with **no documents**, to show what fine-tuning alone can do.

Run **Step 7** after each model: it also saves the trained adapter.

### 5a. Llama-3.1-8B

In [ ]:
!python scripts/04_train_qlora.py --model llama8b
!python scripts/03_run_eval.py --model llama8b --strategy qlora --adapter adapters/llama8b_r16_s42
!python scripts/03_run_eval.py --model llama8b --strategy qlora --adapter adapters/llama8b_r16_s42 --no-docs

### 5b. Mistral-7B

In [ ]:
!python scripts/04_train_qlora.py --model mistral7b
!python scripts/03_run_eval.py --model mistral7b --strategy qlora --adapter adapters/mistral7b_r16_s42
!python scripts/03_run_eval.py --model mistral7b --strategy qlora --adapter adapters/mistral7b_r16_s42 --no-docs

### 5c. Qwen2.5-7B

In [ ]:
!python scripts/04_train_qlora.py --model qwen7b
!python scripts/03_run_eval.py --model qwen7b --strategy qlora --adapter adapters/qwen7b_r16_s42
!python scripts/03_run_eval.py --model qwen7b --strategy qlora --adapter adapters/qwen7b_r16_s42 --no-docs

## Step 6. Results
Run this at any time. It uses whatever has finished so far: the summary table, the statistical tests (once a model has two or more strategies), the TOPSIS ranking and the charts.

In [ ]:
!python scripts/05_analyze.py
!python scripts/06_plot_results.py
import pandas as pd
pd.read_csv('results/summary.csv')

In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('results/figures/*.png')):
    print(f); display(Image(f, width=900))

## Step 7. Save everything to GitHub now
Results already save themselves while a model runs. Run this after Step 6 (to save the summary and charts) or whenever you want to be sure everything is saved.

In [ ]:
from kis.autosave import autosave
import datetime
autosave(f"Main study results ({datetime.date.today()})") or print("Nothing saved: run Setup A first, or check the message above.")